# EK-TTA: ERD-guided Test-Time Adaptation for Motor Imagery EEG

**Experiment E1** — Main LOSO benchmark  
**Experiment E3** — Per-subject ERD% correlation analysis

---
### Data sources (tự động download)
| Dataset | Mô tả | Size |
|---------|-------|------|
| `BNCI2014_001` | BCI IV 2a — 9 subjects, 22ch, 250Hz, 4 classes | ~500 MB |
| `BNCI2014_004` | BCI IV 2b — 9 subjects, 3ch (C3/Cz/C4), 250Hz, 2 classes | ~150 MB |

MOABB tự động download về `~/mne_data/`. **Không cần tải tay.**

---
### Cách chạy
1. `Runtime` → `Change runtime type` → chọn **T4 GPU**  
2. Chạy từng cell theo thứ tự (Shift+Enter)  
3. Kết quả lưu tự động vào Google Drive

In [ ]:
# ── Cell 1: Installation ──────────────────────────────────
!pip install moabb mne torch scipy scikit-learn pandas matplotlib seaborn -q
print("✓ Packages installed")

In [ ]:
# ── Cell 2: Imports & Configuration ──────────────────────
import os, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import welch
from scipy.stats import pearsonr, spearmanr, wilcoxon
from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

import moabb
from moabb.datasets import BNCI2014_001, BNCI2014_004
from moabb.paradigms import MotorImagery
moabb.set_log_level('warning')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ─────────────────────────────────────────────────────────
# CONFIG — Chỉnh tại đây nếu muốn thay đổi hyperparameters
# ─────────────────────────────────────────────────────────
CONFIG = {
    # Sampling
    'sfreq'      : 250,
    'tmin'       : -2.0,   # 2s baseline trước cue (BẮT BUỘC cho ERD)
    'tmax'       : 4.0,    # 4s MI period

    # Index trong epoch dài 6s (1500 samples @ 250Hz)
    # t=-2s → idx=0  |  t=0s → idx=500  |  t=4s → idx=1500
    'baseline_start' : 0,
    'baseline_end'   : 500,    # [-2s, 0s]
    'task_start'     : 625,    # [0.5s, 3.5s] — tránh onset artifact
    'task_end'       : 1375,
    'model_start'    : 500,    # model thấy [0s, 4s] = 1000 samples

    # EEGNet architecture
    'F1'      : 8,
    'D'       : 2,
    'dropout' : 0.5,

    # Training
    'lr'         : 1e-3,
    'epochs'     : 150,
    'batch_size' : 32,
    'patience'   : 25,

    # TTA
    'tta_lr'       : 1e-3,
    'tta_steps'    : 1,       # update steps mỗi trial
    'alpha'        : 1.0,     # weight entropy loss
    'beta'         : 1.0,     # weight ERD loss
    'erd_threshold': -10.0,   # ERD% < threshold → ERD rõ ràng

    # ERD frequency bands
    'mu_low'   : 8,
    'mu_high'  : 12,
    'beta_low' : 13,
    'beta_high': 30,

    # Output — thay đường dẫn nếu cần
    'output_dir': '/content/drive/MyDrive/EK_TTA_results',
    'seed'      : 42,
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
print("✓ Config loaded")

In [ ]:
# ── Cell 3: EEGNet Model ──────────────────────────────────
class EEGNet(nn.Module):
    """
    EEGNet (Lawhern et al. 2018) — lightweight CNN cho EEG.
    ~2300 params với cài đặt mặc định → phù hợp edge device.
    Input shape: (batch, 1, n_channels, n_times)
    """
    def __init__(self, n_classes, n_channels, n_times, F1=8, D=2, dropout=0.5):
        super().__init__()
        F2 = F1 * D

        # Block 1: Temporal convolution
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
        )
        # Block 2: Depthwise spatial convolution
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F2, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )
        # Block 3: Separable convolution
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )
        flat = self._get_flat(n_channels, n_times)
        self.fc = nn.Linear(flat, n_classes)

    def _get_flat(self, n_ch, n_t):
        x = torch.zeros(1, 1, n_ch, n_t)
        return self.block3(self.block2(self.block1(x))).numel()

    def forward(self, x):
        return self.fc(self.block3(self.block2(self.block1(x))).flatten(1))

    def get_bn_params(self):
        """BatchNorm params — dùng cho TTA optimizer."""
        params = []
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                params += [m.weight, m.bias]
        return params

    def set_bn_train(self):
        """Freeze toàn model, chỉ mở BN layers để update."""
        self.eval()
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.train()
                m.weight.requires_grad_(True)
                m.bias.requires_grad_(True)
        return self

print("✓ EEGNet defined")

# Kiểm tra nhanh
_m = EEGNet(n_classes=2, n_channels=22, n_times=1000)
_x = torch.zeros(4, 1, 22, 1000)
print(f"  Output shape: {_m(_x).shape}")
print(f"  Total params: {sum(p.numel() for p in _m.parameters()):,}")

In [ ]:
# ── Cell 4: ERD Module ────────────────────────────────────
def compute_erd(epoch_np, c3_idx, c4_idx, cfg):
    """
    Tính ERD% từ raw EEG (KHÔNG cần nhãn).

    Công thức:
        ERD% = (P_task - P_rest) / P_rest × 100
        Âm = ERD xảy ra = motor cortex đang hoạt động

    Args:
        epoch_np : (n_channels, n_times_full) numpy array
        c3_idx, c4_idx : channel indices
        cfg : CONFIG dict

    Returns:
        dict: C3_mu, C3_beta, C3_comp, C4_mu, C4_beta, C4_comp
    """
    sfreq   = cfg['sfreq']
    nperseg = min(128, (cfg['baseline_end'] - cfg['baseline_start']) // 2)
    erd     = {}

    for name, idx in [('C3', c3_idx), ('C4', c4_idx)]:
        baseline = epoch_np[idx, cfg['baseline_start']:cfg['baseline_end']]
        task     = epoch_np[idx, cfg['task_start']:cfg['task_end']]

        freqs, psd_b = welch(baseline, fs=sfreq, nperseg=nperseg)
        _,     psd_t = welch(task,     fs=sfreq, nperseg=nperseg)

        mu_m   = (freqs >= cfg['mu_low'])   & (freqs <= cfg['mu_high'])
        beta_m = (freqs >= cfg['beta_low']) & (freqs <= cfg['beta_high'])

        for band, mask in [('mu', mu_m), ('beta', beta_m)]:
            pb = np.mean(psd_b[mask]) + 1e-10
            pt = np.mean(psd_t[mask])
            erd[f'{name}_{band}'] = (pt - pb) / pb * 100

        erd[f'{name}_comp'] = (erd[f'{name}_mu'] + erd[f'{name}_beta']) / 2

    return erd


# Sanity check: ERD trên synthetic signal
_epoch = np.random.randn(22, 1500) * 10
# Thêm power ở mu band vào baseline của C3 (giả lập rest)
_epoch[7, 0:500] += 5 * np.sin(2 * np.pi * 10 * np.linspace(0, 2, 500))
_erd = compute_erd(_epoch, c3_idx=7, c4_idx=11, cfg=CONFIG)
print("✓ ERD module defined")
print(f"  Synthetic check — C3_mu ERD%: {_erd['C3_mu']:.1f}%")
print(f"  (âm = ERD xảy ra, dương = không có ERD)")

In [ ]:
# ── Cell 5: Loss Functions ────────────────────────────────
def entropy_loss(logits):
    """
    TENT loss: Shannon entropy minimization.
    Minimize H(p) → model confident hơn.
    Vấn đề: fail khi model confident SAI.
    """
    p = F.softmax(logits, dim=-1)
    return -(p * F.log_softmax(logits, dim=-1)).sum(-1).mean()


def erd_consistency_loss(logits, erd, left_idx, right_idx, threshold):
    """
    ERD Consistency Loss — CORE CONTRIBUTION.

    Logic neurophysiological:
        Predict left hand  → C4 (contralateral) phải ERD âm
        Predict right hand → C3 (contralateral) phải ERD âm

    Hinge-style: chỉ penalize khi ERD không đủ âm.
        relu(ERD - threshold):
            ERD = -30% (mạnh) → relu(-30-(-10)) = relu(-20) = 0  (no penalty)
            ERD =  +5% (không có) → relu(5-(-10)) = relu(15) = 15 (penalize)

    Ưu điểm vs entropy:
        - Không cần model confident → fail-safe
        - Oracle từ raw signal, độc lập với model weights
    """
    probs   = F.softmax(logits, dim=-1)
    p_left  = probs[0, left_idx]
    p_right = probs[0, right_idx]

    c4 = torch.tensor(erd['C4_comp'], dtype=torch.float32, device=logits.device)
    c3 = torch.tensor(erd['C3_comp'], dtype=torch.float32, device=logits.device)

    return p_left * F.relu(c4 - threshold) + p_right * F.relu(c3 - threshold)


print("✓ Loss functions defined")

In [ ]:
# ── Cell 6: TTA Methods ───────────────────────────────────
class NoAdapt:
    """Baseline: không adapt, predict thẳng từ frozen model."""
    def __init__(self, model, cfg):
        self.model = model
    def predict(self, x_raw_np, x_model_t):
        self.model.eval()
        with torch.no_grad():
            return self.model(x_model_t).argmax(-1).item(), None


class TENTMethod:
    """TENT: update BatchNorm params bằng entropy minimization."""
    def __init__(self, model, cfg):
        self.model = model.set_bn_train()
        self.opt   = torch.optim.Adam(model.get_bn_params(), lr=cfg['tta_lr'])
        self.cfg   = cfg
    def predict(self, x_raw_np, x_model_t):
        self.model.set_bn_train()
        for _ in range(self.cfg['tta_steps']):
            loss = entropy_loss(self.model(x_model_t))
            self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.model.eval()
        with torch.no_grad():
            return self.model(x_model_t).argmax(-1).item(), None


class EKTTAMethod:
    """EK-TTA: ERD-guided adaptation (proposed method)."""
    def __init__(self, model, cfg, c3_idx, c4_idx, left_idx, right_idx):
        self.model     = model.set_bn_train()
        self.opt       = torch.optim.Adam(model.get_bn_params(), lr=cfg['tta_lr'])
        self.cfg       = cfg
        self.c3_idx    = c3_idx
        self.c4_idx    = c4_idx
        self.left_idx  = left_idx
        self.right_idx = right_idx
    def predict(self, x_raw_np, x_model_t):
        # Bước 1: tính ERD từ raw signal (không cần GPU, < 1ms)
        erd = compute_erd(x_raw_np[0], self.c3_idx, self.c4_idx, self.cfg)
        # Bước 2: update BN bằng ERD consistency loss
        self.model.set_bn_train()
        for _ in range(self.cfg['tta_steps']):
            logits = self.model(x_model_t)
            loss   = self.cfg['beta'] * erd_consistency_loss(
                logits, erd, self.left_idx, self.right_idx,
                self.cfg['erd_threshold'])
            self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.model.eval()
        with torch.no_grad():
            return self.model(x_model_t).argmax(-1).item(), erd


class EKPlusTENTMethod:
    """EK + TENT: α * L_entropy + β * L_ERD."""
    def __init__(self, model, cfg, c3_idx, c4_idx, left_idx, right_idx):
        self.model     = model.set_bn_train()
        self.opt       = torch.optim.Adam(model.get_bn_params(), lr=cfg['tta_lr'])
        self.cfg       = cfg
        self.c3_idx    = c3_idx
        self.c4_idx    = c4_idx
        self.left_idx  = left_idx
        self.right_idx = right_idx
    def predict(self, x_raw_np, x_model_t):
        erd = compute_erd(x_raw_np[0], self.c3_idx, self.c4_idx, self.cfg)
        self.model.set_bn_train()
        for _ in range(self.cfg['tta_steps']):
            logits = self.model(x_model_t)
            loss   = (self.cfg['alpha'] * entropy_loss(logits) +
                      self.cfg['beta']  * erd_consistency_loss(
                          logits, erd, self.left_idx, self.right_idx,
                          self.cfg['erd_threshold']))
            self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.model.eval()
        with torch.no_grad():
            return self.model(x_model_t).argmax(-1).item(), erd


print("✓ TTA methods defined: NoAdapt | TENT | EK-TTA | EK+TENT")

In [ ]:
# ── Cell 7: Data Loading (MOABB) ──────────────────────────
# Channel names chuẩn
CHANNEL_NAMES = {
    'BNCI2014_001': [
        'Fz','FC3','FC1','FCz','FC2','FC4',
        'C5','C3','C1','Cz','C2','C4','C6',
        'CP3','CP1','CPz','CP2','CP4',
        'P1','Pz','P2','POz'
    ],
    'BNCI2014_004': ['C3', 'Cz', 'C4'],
}


def load_mi_dataset(ds_name, cfg):
    """
    Load dataset qua MOABB với baseline period [-2s, 0s].
    tmin=-2.0 là điểm khác biệt so với các paper khác
    (họ chỉ lấy [0, 4s], bỏ mất baseline cần cho ERD%).

    Returns:
        subject_data : dict {subj: {X_full, X_model, y, c3_idx, c4_idx, ...}}
        label_info   : {left_idx, right_idx, le}
    """
    print(f"\n{'='*55}")
    print(f"  Loading {ds_name} via MOABB...")
    print(f"  (lần đầu sẽ download ~500MB, kiên nhẫn chờ)")

    dataset  = BNCI2014_001() if ds_name == 'BNCI2014_001' else BNCI2014_004()
    paradigm = MotorImagery(
        events   = ['left_hand', 'right_hand'],
        n_classes= 2,
        fmin=0.5, fmax=45.0,
        tmin=cfg['tmin'], tmax=cfg['tmax'],
        resample=cfg['sfreq'],
    )

    X, y, meta = paradigm.get_data(dataset)
    # X: (n_trials, n_channels, n_times_full=1500)

    print(f"  X shape  : {X.shape}")
    print(f"  Duration : {X.shape[2]/cfg['sfreq']:.1f}s per trial")
    print(f"  Subjects : {sorted(meta['subject'].unique())}")

    le     = LabelEncoder()
    y_enc  = le.fit_transform(y)
    classes = list(le.classes_)
    left_idx  = classes.index('left_hand')
    right_idx = classes.index('right_hand')
    print(f"  Encoding : left_hand={left_idx}, right_hand={right_idx}")

    ch_names = CHANNEL_NAMES[ds_name]
    c3_idx   = ch_names.index('C3')
    c4_idx   = ch_names.index('C4')
    print(f"  C3 idx={c3_idx}, C4 idx={c4_idx}")

    subject_data = {}
    ms = cfg['model_start']
    for subj in sorted(meta['subject'].unique()):
        mask = meta['subject'] == subj
        Xs   = X[mask]
        subject_data[subj] = {
            'X_full'   : Xs,              # (n_trials, n_ch, 1500) — cho ERD
            'X_model'  : Xs[:, :, ms:],   # (n_trials, n_ch, 1000) — cho model
            'y'        : y_enc[mask],
            'c3_idx'   : c3_idx,
            'c4_idx'   : c4_idx,
            'left_idx' : left_idx,
            'right_idx': right_idx,
        }

    return subject_data, {'left_idx': left_idx, 'right_idx': right_idx, 'le': le}


print("✓ Data loading function defined")

In [ ]:
# ── Cell 8: Training Pipeline ─────────────────────────────
def train_backbone(X_tr, y_tr, n_ch, n_t, cfg, device):
    """Train EEGNet trên source subjects (LOSO training phase)."""
    model = EEGNet(n_classes=2, n_channels=n_ch, n_times=n_t,
                   F1=cfg['F1'], D=cfg['D'],
                   dropout=cfg['dropout']).to(device)

    X_t   = torch.FloatTensor(X_tr[:, np.newaxis]).to(device)
    y_t   = torch.LongTensor(y_tr).to(device)
    loader = DataLoader(TensorDataset(X_t, y_t),
                        batch_size=cfg['batch_size'], shuffle=True)

    opt  = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg['epochs'])
    crit = nn.CrossEntropyLoss()

    best_loss, best_state, pat = float('inf'), None, 0

    for ep in range(cfg['epochs']):
        model.train()
        total = sum(
            (opt.zero_grad(), loss := crit(model(xb), yb),
             loss.backward(), opt.step(), loss.item())[-1]
            for xb, yb in loader
        )
        sch.step()
        avg = total / len(loader)
        if avg < best_loss:
            best_loss, pat = avg, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            pat += 1
            if pat >= cfg['patience']: break

    model.load_state_dict(best_state)
    return model.eval()


print("✓ Training pipeline defined")

In [ ]:
# ── Cell 9: LOSO Evaluation Loop ─────────────────────────
def run_loso(subject_data, cfg, device, ds_name):
    """
    LOSO evaluation: 9 folds × 4 methods.
    Thu thập đồng thời E1 (accuracy) và E3 (ERD% per subject).
    """
    subjects = sorted(subject_data.keys())
    rows     = []

    for i, test_subj in enumerate(subjects):
        print(f"\n  Fold {i+1}/{len(subjects)} — Test subject: {test_subj}")

        info   = subject_data[test_subj]
        c3, c4 = info['c3_idx'], info['c4_idx']
        li, ri = info['left_idx'], info['right_idx']

        # Source data: tất cả trừ test subject
        src_X = np.concatenate([subject_data[s]['X_model']
                                  for s in subjects if s != test_subj])
        src_y = np.concatenate([subject_data[s]['y']
                                  for s in subjects if s != test_subj])
        n_ch, n_t = src_X.shape[1], src_X.shape[2]

        # Train backbone
        print(f"    Training on {src_X.shape[0]} trials...", end=" ")
        model      = train_backbone(src_X, src_y, n_ch, n_t, cfg, device)
        orig_state = copy.deepcopy(model.state_dict())
        print("done")

        X_full  = info['X_full']   # (n_trials, n_ch, 1500)
        X_model = info['X_model']  # (n_trials, n_ch, 1000)
        y_true  = info['y']

        # ── No-adapt ────────────────────────────────────────
        model.load_state_dict(orig_state); model.eval()
        with torch.no_grad():
            xt = torch.FloatTensor(X_model[:, np.newaxis]).to(device)
            preds_na = model(xt).argmax(-1).cpu().numpy()

        # ── TENT ────────────────────────────────────────────
        model.load_state_dict(orig_state)
        tent_m = TENTMethod(model, cfg)
        preds_tent = []
        for j in range(len(X_model)):
            xt = torch.FloatTensor(X_model[j:j+1, np.newaxis]).to(device)
            p, _ = tent_m.predict(None, xt)
            preds_tent.append(p)
        preds_tent = np.array(preds_tent)

        # ── EK-TTA ──────────────────────────────────────────
        model.load_state_dict(orig_state)
        ek_m = EKTTAMethod(model, cfg, c3, c4, li, ri)
        preds_ek, erds_ek = [], []
        for j in range(len(X_model)):
            xt = torch.FloatTensor(X_model[j:j+1, np.newaxis]).to(device)
            p, erd = ek_m.predict(X_full[j:j+1], xt)
            preds_ek.append(p); erds_ek.append(erd)
        preds_ek = np.array(preds_ek)

        # ── EK + TENT ────────────────────────────────────────
        model.load_state_dict(orig_state)
        ekt_m = EKPlusTENTMethod(model, cfg, c3, c4, li, ri)
        preds_ekt, erds_ekt = [], []
        for j in range(len(X_model)):
            xt = torch.FloatTensor(X_model[j:j+1, np.newaxis]).to(device)
            p, erd = ekt_m.predict(X_full[j:j+1], xt)
            preds_ekt.append(p); erds_ekt.append(erd)
        preds_ekt = np.array(preds_ekt)

        # ERD mean per subject (cho E3)
        erd_means = {f'erd_{k}': np.mean([e[k] for e in erds_ek])
                     for k in erds_ek[0].keys()}

        # Log
        for method, preds in [('no_adapt', preds_na), ('tent', preds_tent),
                               ('ek_tta',  preds_ek), ('ek_tent', preds_ekt)]:
            acc   = accuracy_score(y_true, preds)
            kappa = cohen_kappa_score(y_true, preds)
            row   = {'dataset': ds_name, 'subject': test_subj,
                     'method': method, 'accuracy': acc, 'kappa': kappa}
            row.update(erd_means)
            rows.append(row)
            print(f"    {method:<12} acc={acc:.4f}  kappa={kappa:.4f}")

    return pd.DataFrame(rows)


print("✓ LOSO evaluation function defined")

In [ ]:
# ── Cell 10: RUN ─────────────────────────────────────────
# Mount Google Drive để lưu kết quả
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f"Output dir: {CONFIG['output_dir']}")

all_dfs = []
for ds_name in ['BNCI2014_001', 'BNCI2014_004']:
    subj_data, _ = load_mi_dataset(ds_name, CONFIG)
    df = run_loso(subj_data, CONFIG, device, ds_name)

    path = f"{CONFIG['output_dir']}/{ds_name}_results.csv"
    df.to_csv(path, index=False)
    print(f"\n  ✓ Saved → {path}")
    all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv(f"{CONFIG['output_dir']}/all_results.csv", index=False)
print(f"\n✓ All results saved → {CONFIG['output_dir']}/all_results.csv")
df_all.head(12)

In [ ]:
# ── Cell 11: E1 — Summary Table + Statistical Test ────────
method_order  = ['no_adapt', 'tent', 'ek_tta', 'ek_tent']
method_labels = ['No-adapt', 'TENT', 'EK-TTA (ours)', 'EK+TENT (ours)']

print("TABLE 1 — Main LOSO Results (mean ± std, n=9 subjects)")
print("="*65)

for ds in ['BNCI2014_001', 'BNCI2014_004']:
    df = df_all[df_all['dataset'] == ds]
    print(f"\n  {ds}")
    print(f"  {'Method':<18} {'Acc mean':>10} {'± std':>7} {'Kappa':>8}")
    print(f"  {'-'*48}")
    for m, label in zip(method_order, method_labels):
        sub = df[df['method'] == m]
        print(f"  {label:<18} {sub['accuracy'].mean():>10.4f}"
              f" {sub['accuracy'].std():>7.4f}"
              f" {sub['kappa'].mean():>8.4f}")

print("\n" + "-"*65)
print("STATISTICAL TEST — Wilcoxon signed-rank (EK-TTA vs TENT)")
print("-"*65)
for ds in ['BNCI2014_001', 'BNCI2014_004']:
    df  = df_all[df_all['dataset'] == ds]
    subs = df['subject'].unique()
    ek_a  = [df[(df['subject']==s)&(df['method']=='ek_tta')]['accuracy'].values[0] for s in subs]
    tnt_a = [df[(df['subject']==s)&(df['method']=='tent')]['accuracy'].values[0]   for s in subs]
    delta = np.array(ek_a) - np.array(tnt_a)

    if np.all(delta == 0):
        print(f"  {ds}: No difference detected.")
        continue
    stat, p = wilcoxon(ek_a, tnt_a)
    d = np.mean(delta) / (np.std(delta) + 1e-10)
    sig = "✓ significant" if p < 0.05 else "✗ not significant"
    print(f"  {ds}: W={stat:.0f}, p={p:.4f}, Cohen's d={d:.3f}, "
          f"Δmean={np.mean(delta):+.4f}  → {sig}")

In [ ]:
# ── Cell 12: E1 — Bar Chart ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = ['#888780', '#534AB7', '#0F6E56', '#BA7517']

for ax, ds in zip(axes, ['BNCI2014_001', 'BNCI2014_004']):
    df = df_all[df_all['dataset'] == ds]
    ms = [df[df['method']==m]['accuracy'].mean() for m in method_order]
    ss = [df[df['method']==m]['accuracy'].std()  for m in method_order]

    bars = ax.bar(method_labels, ms, yerr=ss, color=palette,
                  alpha=0.87, capsize=5, width=0.6,
                  error_kw={'lw': 1.5})
    ax.set_title(ds, fontsize=13, fontweight='500')
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_ylim(max(0, min(ms) - 0.08), min(1.0, max(ms) + 0.1))
    ax.grid(axis='y', alpha=0.3)
    for bar, m in zip(bars, ms):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                f'{m:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticklabels(method_labels, rotation=12, ha='right', fontsize=10)

plt.suptitle('E1 — LOSO Accuracy (mean ± std, n=9)', fontsize=13, y=1.02)
plt.tight_layout()
out = f"{CONFIG['output_dir']}/E1_bar_chart.png"
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out}")

In [ ]:
# ── Cell 13: E3 — ERD% vs Δaccuracy (scatter) ────────────
# KEY ANALYSIS:
#   r > 0.5 và p < 0.05 → ERD signal predict được improvement
#   → ERD có thông tin, chỉ loss formulation cần cải tiến (tình huống A/B)
#   r ≈ 0, p > 0.3      → ERD không predict được improvement
#   → Hướng EK genuinely fail (tình huống C)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {'BNCI2014_001': '#534AB7', 'BNCI2014_004': '#0F6E56'}

for ax, ds in zip(axes, ['BNCI2014_001', 'BNCI2014_004']):
    df   = df_all[df_all['dataset'] == ds]
    subs = sorted(df['subject'].unique())

    ek_r  = df[df['method']=='ek_tta'].set_index('subject')
    tnt_r = df[df['method']=='tent'].set_index('subject')

    delta = np.array([ek_r.loc[s,'accuracy'] - tnt_r.loc[s,'accuracy'] for s in subs])
    erd_c = np.array([(ek_r.loc[s,'erd_C3_comp'] + ek_r.loc[s,'erd_C4_comp'])/2 for s in subs])

    r_p, p_p = pearsonr(erd_c, delta)
    r_s, p_s = spearmanr(erd_c, delta)

    ax.scatter(erd_c, delta, c=colors[ds], s=90, zorder=5, alpha=0.9)

    for s, x, y in zip(subs, erd_c, delta):
        ax.annotate(str(s), (x, y), xytext=(5, 4),
                    textcoords='offset points', fontsize=8, color='gray')

    if len(subs) > 2:
        z     = np.polyfit(erd_c, delta, 1)
        x_fit = np.linspace(erd_c.min(), erd_c.max(), 100)
        ax.plot(x_fit, np.poly1d(z)(x_fit), '--', color=colors[ds], alpha=0.5, lw=1.5)

    ax.axhline(0,   color='gray', lw=0.8, alpha=0.5)
    ax.axvline(-10, color='gray', lw=0.8, ls='--', alpha=0.5, label='threshold −10%')

    ax.set_xlabel('Mean ERD% (composite C3+C4)', fontsize=11)
    ax.set_ylabel('Δ Accuracy  (EK-TTA − TENT)', fontsize=11)
    ax.set_title(
        f'{ds}\nPearson r={r_p:.2f} (p={p_p:.3f})  |  Spearman ρ={r_s:.2f} (p={p_s:.3f})',
        fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle('E3 — Does ERD% predict EK improvement?', fontsize=13, y=1.02)
plt.tight_layout()
out = f"{CONFIG['output_dir']}/E3_erd_correlation.png"
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out}")

# Group analysis
print("\nGROUP ANALYSIS (BCI-efficient vs BCI-inefficient)")
print(f"{'Group':<28} {'n':>3} {'ERD% mean':>10} {'Δacc mean':>11}")
print("-"*56)
for ds in ['BNCI2014_001', 'BNCI2014_004']:
    df   = df_all[df_all['dataset'] == ds]
    subs = sorted(df['subject'].unique())
    ek_r = df[df['method']=='ek_tta'].set_index('subject')
    tn_r = df[df['method']=='tent'].set_index('subject')
    erd_ = np.array([(ek_r.loc[s,'erd_C3_comp']+ek_r.loc[s,'erd_C4_comp'])/2 for s in subs])
    dac_ = np.array([ek_r.loc[s,'accuracy']-tn_r.loc[s,'accuracy'] for s in subs])

    for label, mask in [('efficient   (ERD < -10%)', erd_ < -10),
                         ('inefficient (ERD ≥ -10%)', erd_ >= -10)]:
        n = mask.sum()
        if n == 0: continue
        print(f"{ds} {label:<22} {n:>3} {erd_[mask].mean():>10.1f}%"
              f" {dac_[mask].mean():>+11.4f}")

## Đọc kết quả

### Cell 11 — Table 1
| Kịch bản | Dấu hiệu | Kết luận |
|----------|----------|----------|
| **EK tốt hơn** | `ek_tta` > `tent`, p < 0.05 | Hướng EK thành công, tiếp tục E2, E4, E5 |
| **EK ngang** | `ek_tta` ≈ `tent`, Cohen's d nhỏ | Xem E3 để phân tích cơ chế |
| **EK kém** | `ek_tta` < `tent` | Xem E3 ngay để xác định tình huống A/B/C |

### Cell 13 — E3 Scatter
| Kết quả scatter | Ý nghĩa |
|----------------|---------|
| r > 0.5, p < 0.05 | ERD signal có thông tin → chỉ loss formulation cần điều chỉnh |
| r ≈ 0, p > 0.3 | ERD không correlate → hướng EK cần reconsider |

---
**Files output (Google Drive):**
```
EK_TTA_results/
├── BNCI2014_001_results.csv
├── BNCI2014_004_results.csv
├── all_results.csv
├── E1_bar_chart.png
└── E3_erd_correlation.png
```